In [5]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
print(os.listdir('/content/drive/MyDrive'))

Mounted at /content/drive
['Colab Notebooks', 'derivation_boundaries']


In [6]:
RESULTS_PATH = '/content/drive/MyDrive/derivation_boundaries/A1_A2_logit_lens_clean_llama.json'

import os, json
print("Found:", os.path.exists(RESULTS_PATH))
if os.path.exists(RESULTS_PATH):
    R = json.load(open(RESULTS_PATH))
    print("Steps:", len(R))
    print("Problems done:", sorted({r['problem_idx'] for r in R}))

Found: True
Steps: 792
Problems done: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]


In [7]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

GPU: Tesla T4


In [8]:
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 52.8 MB/s eta 0:00:00


In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
MODEL_NAME = "llama_3.1_8b"
N_LAYERS = 32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map={"": 0})
model.eval()
torch.cuda.empty_cache()
print("Model loaded.")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded.
GPU memory: 5.70 GB


In [2]:
import os, json, subprocess
import numpy as np

if not os.path.exists("/content/RuleArena"):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/SkyRiver-2000/RuleArena.git",
                    "/content/RuleArena"], check=True)

os.chdir("/content/RuleArena/airline")
exec(open("compute_answer.py").read().split("if __name__")[0])
check_base_tables = load_checking_fee()

problems = []
with open("synthesized_problems/comp_0.jsonl") as f:
    for line in f:
        problems.append(json.loads(line))

def compute_gt(info):
    total, gt_info = compute_answer(
        base_price=info["base_price"], direction=info["direction"],
        routine=info["routine"], customer_class=info["customer_class"],
        bag_list=[{"id": 0, "name": "ticket", "size": [0,0,0], "weight": 0}] + info["bag_list"],
        check_base_tables=check_base_tables,
    )
    return total, gt_info

print("Problems:", len(problems), "| GT problem 0:", compute_gt(problems[0]["info"])[0])

Problems: 100 | GT problem 0: 1445


In [3]:
import re

with open("/content/RuleArena/airline/reference_rules.txt") as f:
    REFERENCE_RULES = f.read()
print(f"Rules loaded: {len(REFERENCE_RULES)} chars")


def generate_steps(problem, gt_info):
    info = problem["info"]
    steps = []

    for i, bag in enumerate(info["bag_list"]):
        bag_id = bag["id"]
        dims = bag["size"]
        weight = bag["weight"]
        dim_sum = sum(dims)

        correct_oversize_fee = gt_info["oversize"][i]
        correct_overweight_fee = gt_info["overweight"][i]
        correct_base_fee = gt_info["base"][i]

        # ── ELEMENTARY: pure arithmetic and comparison only ────

        steps.append({
            "step_id": f"bag{bag_id}_EA",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compute dimension sum",
            "system": "You are a calculator. Answer with just the number.",
            "user": f"What is {dims[0]} + {dims[1]} + {dims[2]}? Answer with just the number.",
            "correct_answer": str(dim_sum),
        })

        expected_oversize = "OVERSIZE" if dim_sum > 62 else "NOT OVERSIZE"
        steps.append({
            "step_id": f"bag{bag_id}_EB",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compare sum to oversize threshold",
            "system": "Answer with exactly OVERSIZE or NOT OVERSIZE.",
            "user": (
                f"A bag has total dimensions of {dim_sum} inches. "
                f"The oversize threshold is 62 inches. "
                f"Is this bag OVERSIZE or NOT OVERSIZE?"
            ),
            "correct_answer": expected_oversize,
        })

        expected_overweight = "OVERWEIGHT" if weight > 50 else "NOT OVERWEIGHT"
        steps.append({
            "step_id": f"bag{bag_id}_EC",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compare weight to overweight threshold",
            "system": "Answer with exactly OVERWEIGHT or NOT OVERWEIGHT.",
            "user": (
                f"A bag weighs {weight} lbs. "
                f"The overweight threshold is 50 lbs. "
                f"Is this bag OVERWEIGHT or NOT OVERWEIGHT?"
            ),
            "correct_answer": expected_overweight,
        })

        # ── ED: table read oversize — elementary if fee>0, boundary if fee=0
        if dim_sum > 62:
            if dim_sum <= 65:
                bracket = "Over 62 inches to 65 inches"
            else:
                bracket = "Over 65 inches to 115 inches"
            steps.append({
                "step_id": f"bag{bag_id}_ED",
                "bag_id": bag_id,
                "type": "elementary" if correct_oversize_fee > 0 else "boundary",
                "description": "Read oversize fee from table given bracket"
                               + (" (complimentary rule applies → boundary)" if correct_oversize_fee == 0 else ""),
                "system": "You are an airline fee calculator. Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag is in bracket: {bracket}.\n\n"
                    f"OVERSIZE FEE TABLE:\n"
                    f"Between U.S., Puerto Rico, U.S. Virgin Islands and Canada:\n"
                    f"- Over 62 inches to 65 inches: $30\n"
                    f"- Over 65 inches to 115 inches: $200\n\n"
                    f"What is the oversize fee for this passenger? "
                    f"Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_oversize_fee}",
            })

        # ── EE: table read overweight — elementary if fee>0, boundary if fee=0
        if weight > 50:
            if weight <= 53:
                wbracket = "Over 50 lbs to 53 lbs"
            elif weight <= 70:
                wbracket = "Over 53 lbs to 70 lbs"
            else:
                wbracket = "Over 70 lbs to 100 lbs"
            steps.append({
                "step_id": f"bag{bag_id}_EE",
                "bag_id": bag_id,
                "type": "elementary" if correct_overweight_fee > 0 else "boundary",
                "description": "Read overweight fee from table given bracket"
                               + (" (complimentary rule applies → boundary)" if correct_overweight_fee == 0 else ""),
                "system": "You are an airline fee calculator. Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag is in bracket: {wbracket}.\n\n"
                    f"OVERWEIGHT FEE TABLE:\n"
                    f"- Over 50 lbs to 53 lbs: $30\n"
                    f"- Over 53 lbs to 70 lbs: $100\n"
                    f"- Over 70 lbs to 100 lbs: $200\n\n"
                    f"What is the overweight fee for this passenger? "
                    f"Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_overweight_fee}",
            })

        # ── BOUNDARY: everything requiring rule knowledge ──────

        steps.append({
            "step_id": f"bag{bag_id}_B0",
            "bag_id": bag_id,
            "type": "boundary",
            "description": "Is bag complimentary? (class + bag number + route rule)",
            "system": "Answer with exactly YES or NO.",
            "user": (
                f"Passenger class: {gt_info['customer_class']}\n"
                f"Route: {gt_info['routine']} domestic\n"
                f"This is bag number {bag_id} "
                f"(the {['1st','2nd','3rd','4th','5th'][bag_id-1]} checked bag).\n\n"
                f"FEE TABLE:\n{REFERENCE_RULES}\n\n"
                f"Is this bag complimentary (free) for this passenger? "
                f"Answer with exactly YES or NO."
            ),
            "correct_answer": "YES" if correct_base_fee == 0 else "NO",
        })

        if dim_sum > 62:
            steps.append({
                "step_id": f"bag{bag_id}_B1",
                "bag_id": bag_id,
                "type": "boundary",
                "description": "Raw dimensions → oversize fee (derive bracket + lookup)",
                "system": "Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag dimensions: {dims[0]} x {dims[1]} x {dims[2]} inches.\n\n"
                    f"FEE TABLE:\n{REFERENCE_RULES}\n\n"
                    f"What is the oversize fee for this bag? "
                    f"Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_oversize_fee}",
            })

        if weight > 50:
            steps.append({
                "step_id": f"bag{bag_id}_B2",
                "bag_id": bag_id,
                "type": "boundary",
                "description": "Raw weight → overweight fee (derive bracket + lookup)",
                "system": "Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag weight: {weight} lbs.\n\n"
                    f"FEE TABLE:\n{REFERENCE_RULES}\n\n"
                    f"What is the overweight fee for this bag? "
                    f"Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_overweight_fee}",
            })

    return steps


def extract_answer(response, step_type):
    response = response.strip()
    for keyword in ["NOT OVERSIZE", "NOT OVERWEIGHT", "OVERSIZE", "OVERWEIGHT"]:
        if keyword in response.upper():
            return keyword
    if response.upper().startswith("YES"):
        return "YES"
    if response.upper().startswith("NO"):
        return "NO"
    match = re.search(r'\$(\d+)', response)
    if match:
        return f"${match.group(1)}"
    nums = re.findall(r'\b\d+\b', response)
    if nums:
        return nums[0]
    return response.strip()


def build_prompt(step):
    messages = [
        {"role": "system", "content": step["system"]},
        {"role": "user", "content": step["user"]},
    ]
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)


gt, gt_info = compute_gt(problems[0]["info"])
steps = generate_steps(problems[0], gt_info)
print(f"Problem 0: {len(steps)} steps — "
      f"{sum(1 for s in steps if s['type']=='elementary')} elementary, "
      f"{sum(1 for s in steps if s['type']=='boundary')} boundary")

Rules loaded: 19583 chars
Problem 0: 36 steps — 23 elementary, 13 boundary


In [4]:
from google.colab import drive
drive.mount('/content/drive')

RESULTS_PATH = '/content/drive/MyDrive/derivation_boundaries/A1_A2_logit_lens_clean_llama.json'
R = json.load(open(RESULTS_PATH))
print("Existing steps:", len(R))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Existing steps: 792


In [5]:
import torch
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

@torch.no_grad()
def answer_and_lens_fast(step, max_new_tokens=8):
    prompt = build_prompt(step)
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = enc["input_ids"].shape[-1]

    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        output_hidden_states=True, return_dict_in_generate=True,
    )
    answer_ids = out.sequences[0][prompt_len:]
    answer_text = tokenizer.decode(answer_ids, skip_special_tokens=True)

    ans_tokens = [tokenizer.decode([t]) for t in answer_ids]
    read_off = 0
    for j, t in enumerate(ans_tokens):
        if any(ch.isdigit() for ch in t):
            read_off = j
            break

    target_id = answer_ids[read_off].item()
    step_hs = out.hidden_states[read_off]
    norm, head = model.model.norm, model.lm_head

    commit_top1, commit_p90, probs = None, None, []
    for L in range(1, N_LAYERS + 1):
        h = step_hs[L][0, -1]
        p = torch.softmax(head(norm(h)).float(), dim=-1)
        pt = p[target_id].item()
        probs.append(round(pt, 4))
        if commit_top1 is None and torch.argmax(p).item() == target_id:
            commit_top1 = L
        if commit_p90 is None and pt > 0.9:
            commit_p90 = L
        if L == N_LAYERS:
            v, i = torch.topk(p, 5)
            top5 = [(tokenizer.decode([a]), round(float(b), 4)) for a, b in zip(i.tolist(), v.tolist())]

    return {
        "answer_text": answer_text,
        "readout_token": tokenizer.decode([target_id]),
        "commit_top1": commit_top1 or N_LAYERS,
        "commit_p90": commit_p90 or N_LAYERS,
        "probs_by_layer": probs,
        "top5_final": top5,
    }

print("fast version ready")

fast version ready


In [6]:
import time

saved = {(r["problem_idx"], r["step_id"]): r for r in R}
gt, gt_info = compute_gt(problems[0]["info"])
steps = generate_steps(problems[0], gt_info)

t0 = time.time()
n = 0
for s in steps[:4]:
    key = (0, s["step_id"])
    if key not in saved:
        continue
    old = saved[key]
    new = answer_and_lens_fast(s)
    same = (old["commit_top1"] == new["commit_top1"] and
            old["readout_token"] == new["readout_token"] and
            old["commit_p90"] == new["commit_p90"])
    print(f"{s['step_id']:12} old={old['commit_top1']:2}  new={new['commit_top1']:2}  MATCH={same}")
    n += 1

print(f"\n{(time.time()-t0)/max(n,1):.1f} sec/step   (slow version was ~25 sec/step)")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


bag1_EA      old=23  new=23  MATCH=True
bag1_EB      old=32  new=32  MATCH=True
bag1_EC      old=31  new=31  MATCH=True
bag1_B0      old=32  new=32  MATCH=True

8.0 sec/step   (slow version was ~25 sec/step)


In [7]:
for sid in ["bag2_ED", "bag2_B2"]:
    s = next(x for x in generate_steps(problems[0], compute_gt(problems[0]["info"])[1])
             if x["step_id"] == sid)
    old = saved[(0, sid)]
    new = answer_and_lens_fast(s)
    print(f"{sid}")
    print(f"  commit  old={old['commit_top1']} new={new['commit_top1']}  match={old['commit_top1']==new['commit_top1']}")
    print(f"  readout old={old['readout_token']!r} new={new['readout_token']!r}")
    print(f"  top5 old={old['top5_final'][:3]}")
    print(f"  top5 new={new['top5_final'][:3]}")
    print()

bag2_ED
  commit  old=25 new=25  match=True
  readout old='200' new='200'
  top5 old=[['200', 0.9994], ['0', 0.0003], ['30', 0.0001]]
  top5 new=[('200', 0.9994), ('0', 0.0003), ('30', 0.0001)]

bag2_B2
  commit  old=23 new=23  match=True
  readout old='30' new='30'
  top5 old=[['30', 0.6382], ['100', 0.3416], ['200', 0.008]]
  top5 new=[('30', 0.6658), ('100', 0.3145), ('200', 0.0084)]



In [ ]:
import time

START_IDX = 0
END_IDX = 60

RESULTS_PATH_V2 = '/content/drive/MyDrive/derivation_boundaries/A1_A2_logit_lens_clean_llama_v2.json'

if os.path.exists(RESULTS_PATH_V2):
    all_results = json.load(open(RESULTS_PATH_V2))
    done = {r["problem_idx"] for r in all_results}
    print(f"Resuming v2 — {len(done)} problems, {len(all_results)} steps.")
else:
    all_results = []
    done = set()
    print("Starting v2 fresh.")

t_start = time.time()

for idx in range(START_IDX, END_IDX):
    if idx in done:
        continue
    try:
        gt, gt_info = compute_gt(problems[idx]["info"])
        steps = generate_steps(problems[idx], gt_info)

        for s in steps:
            r = answer_and_lens_fast(s)
            pred = extract_answer(r["answer_text"], s["type"])
            all_results.append({
                "problem_idx": idx,
                "step_id": s["step_id"],
                "type": s["type"],
                "description": s["description"],
                "correct_answer": s["correct_answer"],
                "predicted": pred,
                "correct": pred.strip().upper() == s["correct_answer"].strip().upper(),
                "readout_token": r["readout_token"],
                "commit_top1": r["commit_top1"],
                "commit_p90": r["commit_p90"],
                "probs_by_layer": r["probs_by_layer"],
                "top5_final": r["top5_final"],
            })

        with open(RESULTS_PATH_V2, "w") as f:
            json.dump(all_results, f)

        def sfx(x): return x["step_id"].split("_")[1]
        giv = [x for x in all_results if sfx(x) in ("ED","EE")]
        der = [x for x in all_results if sfx(x) in ("B1","B2")]
        mins = (time.time() - t_start) / 60
        n_done = len([i for i in range(START_IDX, idx+1) if i not in done])
        eta = mins / max(n_done, 1) * (END_IDX - idx - 1)
        print(f"[{idx+1}/60] steps={len(all_results)}  "
              f"GIVEN acc={np.mean([x['correct'] for x in giv]):.1%} L={np.mean([x['commit_top1'] for x in giv]):.2f}  |  "
              f"DERIVED acc={np.mean([x['correct'] for x in der]):.1%} L={np.mean([x['commit_top1'] for x in der]):.2f}  "
              f"({mins:.0f}m, ~{eta:.0f}m left)")

    except Exception as e:
        print(f"  ERROR problem {idx}: {e}")
        torch.cuda.empty_cache()
        continue

print(f"\nDone. {len(all_results)} steps, {(time.time()-t_start)/60:.0f} min.")

Starting v2 fresh.
[1/60] steps=36  GIVEN acc=100.0% L=23.75  |  DERIVED acc=37.5% L=23.00  (7m, ~423m left)
[2/60] steps=72  GIVEN acc=93.8% L=23.88  |  DERIVED acc=31.2% L=23.00  (14m, ~417m left)
[3/60] steps=108  GIVEN acc=91.7% L=23.79  |  DERIVED acc=37.5% L=23.00  (22m, ~410m left)
[4/60] steps=144  GIVEN acc=93.8% L=23.81  |  DERIVED acc=37.5% L=23.00  (29m, ~403m left)
[5/60] steps=180  GIVEN acc=85.0% L=23.93  |  DERIVED acc=30.0% L=23.00  (36m, ~395m left)
[6/60] steps=216  GIVEN acc=87.5% L=23.85  |  DERIVED acc=35.4% L=23.04  (43m, ~388m left)
[7/60] steps=252  GIVEN acc=85.7% L=23.80  |  DERIVED acc=37.5% L=23.04  (50m, ~381m left)
[8/60] steps=288  GIVEN acc=84.4% L=23.70  |  DERIVED acc=42.2% L=23.03  (58m, ~374m left)
[9/60] steps=324  GIVEN acc=86.1% L=23.69  |  DERIVED acc=43.1% L=23.03  (65m, ~367m left)
[10/60] steps=360  GIVEN acc=87.5% L=23.71  |  DERIVED acc=42.5% L=23.02  (72m, ~360m left)
[11/60] steps=396  GIVEN acc=88.6% L=23.72  |  DERIVED acc=42.0% L=23.02

In [ ]:
import json, numpy as np
from scipy import stats

RESULTS_PATH_V2 = '/content/drive/MyDrive/derivation_boundaries/A1_A2_logit_lens_clean_llama_v2.json'
R2 = json.load(open(RESULTS_PATH_V2))

def sfx(r): return r["step_id"].split("_")[1]

print(f"Total steps: {len(R2)}   problems: {len(set(r['problem_idx'] for r in R2))}\n")

given   = [r for r in R2 if sfx(r) in ("ED","EE")]
derived = [r for r in R2 if sfx(r) in ("B1","B2")]

for name, g in [("BRACKET GIVEN   (ED/EE)", given), ("BRACKET DERIVED (B1/B2)", derived)]:
    c = [r["commit_top1"] for r in g]
    print(f"{name}")
    print(f"   n = {len(g)}")
    print(f"   commit layer = {np.mean(c):.2f}  (median {np.median(c):.0f}, sd {np.std(c):.2f})")
    print(f"   accuracy     = {np.mean([r['correct'] for r in g]):.1%}")
    print()

a = np.array([r["commit_top1"] for r in derived])
b = np.array([r["commit_top1"] for r in given])
U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
print("─"*52)
print(f"commit difference (derived - given) = {a.mean()-b.mean():+.2f} layers")
print(f"p = {p:.2e}   effect size = {2*U/(len(a)*len(b))-1:+.3f}")

acc_g = np.mean([r['correct'] for r in given])
acc_d = np.mean([r['correct'] for r in derived])
chi, pa = stats.fisher_exact([
    [sum(r['correct'] for r in derived), len(derived)-sum(r['correct'] for r in derived)],
    [sum(r['correct'] for r in given),   len(given)-sum(r['correct'] for r in given)]])
print(f"accuracy difference = {acc_d-acc_g:+.1%}   Fisher exact p = {pa:.2e}")
print("─"*52)

In [ ]:
from collections import Counter

def norm(x): return str(x).strip().lstrip("$")

def analyse(group, label):
    wrong = [r for r in group if not r["correct"]]
    live, ranks, top1p, corrp = 0, [], [], []
    for r in wrong:
        want = norm(r["correct_answer"])
        toks = [norm(t) for t, p in r["top5_final"]]
        prob = [p for t, p in r["top5_final"]]
        top1p.append(prob[0])
        if want in toks:
            live += 1
            i = toks.index(want)
            ranks.append(i+1)
            corrp.append(prob[i])
    print(f"── {label} ──")
    print(f"  wrong: {len(wrong)} / {len(group)}  ({len(wrong)/len(group):.1%})")
    if wrong:
        print(f"  correct answer still in top-5 at final layer: {live}/{len(wrong)} = {live/len(wrong):.1%}")
        if ranks:
            print(f"    rank distribution : {sorted(Counter(ranks).items())}")
            print(f"    its probability   : mean {np.mean(corrp):.3f}")
        print(f"  confidence in WRONG answer: mean {np.mean(top1p):.3f}")
    print()

analyse(derived, "BRACKET DERIVED (B1/B2) — boundary")
analyse(given,   "BRACKET GIVEN   (ED/EE) — control")